# Medicare Utilization Analysis — Data Cleaning

This notebook prepares an analysis-ready county-level dataset by combining:

- **CMS Original Medicare Geographic Variation PUF (2020–2024)** for beneficiary demographics and healthcare utilization
- **2024 ACS 5-Year Data Profiles** for county-level socioeconomic measures

The workflow:

1. Clean and validate CMS county-year data
2. Retrieve and clean ACS county data
3. Compare FIPS coverage and resolve geography mismatches
4. Aggregate CMS 2020–2024 to one row per geography
5. Merge CMS and ACS on 5-digit FIPS
6. Validate and export the final processed dataset

### Key cleaning decisions

- CMS suppression marker `*` is converted to `NULL`; suppressed values are **not** imputed or treated as zero.
- Census negative sentinel values are converted to `NULL`.
- CMS `UNKNOWN` geography rows are excluded.
- Connecticut is excluded from the merged analysis because CMS changes from the historical 8 counties in 2020–2021 to 9 planning regions in 2022–2024, while the 2024 ACS uses the newer geography.
- CMS annual measures are aggregated using Original Medicare beneficiary counts as weights. The beneficiary count itself is summarized as an average annual count.

### Data sources

CMS Medicare Geographic Variation:
https://data.cms.gov/summary-statistics-on-use-and-payments/medicare-geographic-comparisons/medicare-geographic-variation-by-national-state-county

ACS 2024 5-Year Data Profile API:
https://api.census.gov/data/2024/acs/acs5/profile

## 1. Setup

In [1]:
import os

import duckdb
import pandas as pd
import requests
from dotenv import load_dotenv

CMS_PATH = "data/raw/cms/2014-2024 Original Medicare Geographic Variation Public Use File.csv"
OUTPUT_PATH = "data/processed/medicare_acs_analysis.csv"

os.makedirs("data/processed", exist_ok=True)

## 2. CMS Data Exploration and Cleaning

The analysis uses county-level CMS records from **2020–2024**. The fields retained are:

- geography: year, county name, FIPS
- Original Medicare beneficiary count
- average age
- female and male percentages
- inpatient stays per 1,000 beneficiaries
- outpatient visits per 1,000 beneficiaries
- emergency room visits per 1,000 beneficiaries

In [2]:
original_df = pd.read_csv(CMS_PATH, low_memory=False)

print("Shape:", original_df.shape)
print("Geography levels:", original_df["BENE_GEO_LVL"].unique())

county = original_df[original_df["BENE_GEO_LVL"] == "County"]
county.groupby("YEAR").size()

Shape: (36994, 246)
Geography levels: <ArrowStringArray>
['National', 'State', 'County']
Length: 3, dtype: str


YEAR
2014    3194
2015    3195
2016    3194
2017    3194
2018    3194
2019    3194
2020    3194
2021    3196
2022    3197
2023    3197
2024    3197
dtype: int64

### CMS suppression profile

In [3]:
duckdb.sql(f'''
SELECT
    YEAR AS year,
    COUNT(*) FILTER (WHERE BENES_OM_CNT = '*') AS om_count_suppressed,
    COUNT(*) FILTER (WHERE BENE_AVG_AGE = '*') AS avg_age_suppressed,
    COUNT(*) FILTER (WHERE BENE_FEML_PCT = '*') AS female_suppressed,
    COUNT(*) FILTER (WHERE BENE_MALE_PCT = '*') AS male_suppressed,
    COUNT(*) FILTER (WHERE IP_CVRD_STAYS_PER_1000_BENES = '*') AS inpatient_suppressed,
    COUNT(*) FILTER (WHERE OP_VISITS_PER_1000_BENES = '*') AS outpatient_suppressed,
    COUNT(*) FILTER (WHERE ER_VISITS_PER_1000_BENES = '*') AS er_suppressed
FROM read_csv('{CMS_PATH}')
WHERE BENE_GEO_LVL = 'County'
  AND YEAR BETWEEN 2020 AND 2024
GROUP BY YEAR
ORDER BY YEAR;
''')

┌───────┬─────────────────────┬────────────────────┬───────────────────┬─────────────────┬──────────────────────┬───────────────────────┬───────────────┐
│ year  │ om_count_suppressed │ avg_age_suppressed │ female_suppressed │ male_suppressed │ inpatient_suppressed │ outpatient_suppressed │ er_suppressed │
│ int64 │        int64        │       int64        │       int64       │      int64      │        int64         │         int64         │     int64     │
├───────┼─────────────────────┼────────────────────┼───────────────────┼─────────────────┼──────────────────────┼───────────────────────┼───────────────┤
│  2020 │                  52 │                 52 │                54 │              54 │                   58 │                    53 │            55 │
│  2021 │                  54 │                 54 │                56 │              56 │                   61 │                    55 │            56 │
│  2022 │                  52 │                 52 │                54 │    

### Create cleaned CMS view

`*` suppression values are converted to `NULL`. The view also removes `UNKNOWN` geography rows so that only identifiable county/county-equivalent records remain.

In [4]:
duckdb.sql(f'''
CREATE OR REPLACE VIEW cms_clean AS
SELECT
    YEAR AS year,
    BENE_GEO_DESC AS county_name,
    BENE_GEO_CD AS fips,
    CAST(NULLIF(BENES_OM_CNT, '*') AS INTEGER) AS om_beneficiaries,
    CAST(NULLIF(BENE_AVG_AGE, '*') AS INTEGER) AS avg_age,
    CAST(NULLIF(BENE_FEML_PCT, '*') AS DOUBLE) AS female_pct,
    CAST(NULLIF(BENE_MALE_PCT, '*') AS DOUBLE) AS male_pct,
    CAST(NULLIF(IP_CVRD_STAYS_PER_1000_BENES, '*') AS DOUBLE) AS inpatient_stays_per_1000,
    CAST(NULLIF(OP_VISITS_PER_1000_BENES, '*') AS DOUBLE) AS outpatient_visits_per_1000,
    CAST(NULLIF(ER_VISITS_PER_1000_BENES, '*') AS DOUBLE) AS er_visits_per_1000
FROM read_csv('{CMS_PATH}')
WHERE BENE_GEO_LVL = 'County'
  AND YEAR BETWEEN 2020 AND 2024
  AND BENE_GEO_DESC NOT LIKE '%UNKNOWN%';
''')

### Validate cleaned CMS data

In [5]:
duckdb.sql('''
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT fips) AS unique_fips,
    MIN(year) AS first_year,
    MAX(year) AS last_year
FROM cms_clean;
''')

┌───────────┬─────────────┬────────────┬───────────┐
│ row_count │ unique_fips │ first_year │ last_year │
│   int64   │    int64    │   int64    │   int64   │
├───────────┼─────────────┼────────────┼───────────┤
│     15726 │        3154 │       2020 │      2024 │
└───────────┴─────────────┴────────────┴───────────┘

In [6]:
duckdb.sql('''
SELECT
    year,
    fips,
    COUNT(*) AS row_count
FROM cms_clean
GROUP BY year, fips
HAVING COUNT(*) > 1;
''')

┌───────┬─────────┬───────────┐
│ year  │  fips   │ row_count │
│ int64 │ varchar │   int64   │
└───────┴─────────┴───────────┘
            0 rows           

In [7]:
duckdb.sql('''
SELECT
    year,
    county_name,
    fips
FROM cms_clean
WHERE LENGTH(fips) != 5;
''')

┌───────┬─────────────┬─────────┐
│ year  │ county_name │  fips   │
│ int64 │   varchar   │ varchar │
└───────┴─────────────┴─────────┘
             0 rows            

## 3. ACS Data Retrieval and Cleaning

The ACS source is the **2024 ACS 5-Year Data Profile API**. The socioeconomic variables are:

- Median household income — `DP03_0062E`
- Poverty rate — `DP03_0128PE`
- Bachelor's degree or higher, age 25+ — `DP02_0068PE`

The 2024 ACS 5-Year estimates pool survey responses from 2020–2024, so the ACS data already provides one estimate per geography rather than separate annual observations.

In [8]:
load_dotenv()
api_key = os.getenv("CENSUS_API_KEY")

if api_key is None:
    raise ValueError("CENSUS_API_KEY was not found. Check your .env file.")

In [9]:
acs_variables = {
    "median_household_income": "DP03_0062E",
    "poverty_rate": "DP03_0128PE",
    "bachelors_or_higher_pct": "DP02_0068PE",
}

base_url = "https://api.census.gov/data/2024/acs/acs5/profile"

params = {
    "key": api_key,
    "get": ",".join(["NAME"] + list(acs_variables.values())),
    "for": "county:*",
    "in": "state:*",
}

response = requests.get(base_url, params=params, timeout=30)
response.raise_for_status()
data = response.json()

acs_raw = pd.DataFrame(data[1:], columns=data[0])
acs_raw.head()

,NAME,DP03_0062E,DP03_0128PE,DP02_0068PE,state,county
0,"Autauga County, Alabama",72481,11.3,29.1,01,001
1,"Baldwin County, Alabama",78775,10.1,33.7,01,003
2,"Barbour County, Alabama",46042,21.4,10.6,01,005
3,"Bibb County, Alabama",52541,22.5,11.7,01,007
4,"Blount County, Alabama",64190,12.9,15.8,01,009


### Create cleaned ACS view

The Census API returns values as strings. Negative Census sentinel values are converted to `NULL`, and state + county codes are concatenated into a 5-digit FIPS key.

In [10]:
duckdb.register("acs_raw", acs_raw)

duckdb.sql('''
CREATE OR REPLACE VIEW acs_clean AS
SELECT
    NAME AS county_name,
    CASE
        WHEN CAST(DP03_0062E AS INTEGER) < 0 THEN NULL
        ELSE CAST(DP03_0062E AS INTEGER)
    END AS median_household_income,
    CASE
        WHEN CAST(DP03_0128PE AS DOUBLE) < 0 THEN NULL
        ELSE CAST(DP03_0128PE AS DOUBLE)
    END AS poverty_rate,
    CASE
        WHEN CAST(DP02_0068PE AS DOUBLE) < 0 THEN NULL
        ELSE CAST(DP02_0068PE AS DOUBLE)
    END AS bachelors_or_higher_pct,
    state AS state_fips,
    county AS county_fips,
    state || county AS fips
FROM acs_raw;
''')

### Validate ACS data

In [11]:
duckdb.sql('''
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT fips) AS unique_fips,
    COUNT(*) FILTER (WHERE median_household_income IS NULL) AS missing_income,
    COUNT(*) FILTER (WHERE poverty_rate IS NULL) AS missing_poverty,
    COUNT(*) FILTER (WHERE bachelors_or_higher_pct IS NULL) AS missing_education
FROM acs_clean;
''')

┌───────────┬─────────────┬────────────────┬─────────────────┬───────────────────┐
│ row_count │ unique_fips │ missing_income │ missing_poverty │ missing_education │
│   int64   │    int64    │     int64      │      int64      │       int64       │
├───────────┼─────────────┼────────────────┼─────────────────┼───────────────────┤
│      3222 │        3222 │              1 │               0 │                78 │
└───────────┴─────────────┴────────────────┴─────────────────┴───────────────────┘

In [12]:
duckdb.sql('''
SELECT
    county_name,
    state_fips,
    county_fips,
    fips
FROM acs_clean
WHERE LENGTH(fips) != 5;
''')

┌─────────────┬────────────┬─────────────┬─────────┐
│ county_name │ state_fips │ county_fips │  fips   │
│   varchar   │  varchar   │   varchar   │ varchar │
└─────────────┴────────────┴─────────────┴─────────┘
                       0 rows                     

## 4. CMS–ACS Geographic Coverage Comparison

Before merging, compare distinct FIPS coverage between the two datasets. This identifies geographies that cannot be matched directly and surfaces geography-vintage differences.

In [13]:
duckdb.sql('''
SELECT
    (SELECT COUNT(DISTINCT fips) FROM cms_clean) AS cms_unique_fips,
    (SELECT COUNT(DISTINCT fips) FROM acs_clean) AS acs_unique_fips;
''')

┌─────────────────┬─────────────────┐
│ cms_unique_fips │ acs_unique_fips │
│      int64      │      int64      │
├─────────────────┼─────────────────┤
│            3154 │            3222 │
└─────────────────┴─────────────────┘

### ACS geographies without a CMS match

In [14]:
duckdb.sql('''
SELECT
    acs.county_name,
    acs.state_fips,
    acs.county_fips,
    acs.fips
FROM acs_clean AS acs
LEFT JOIN (
    SELECT DISTINCT fips
    FROM cms_clean
) AS cms
    ON acs.fips = cms.fips
WHERE cms.fips IS NULL
ORDER BY acs.fips
LIMIT 10;
''')

┌────────────────────────────────────────────┬────────────┬─────────────┬─────────┐
│                county_name                 │ state_fips │ county_fips │  fips   │
│                  varchar                   │  varchar   │   varchar   │ varchar │
├────────────────────────────────────────────┼────────────┼─────────────┼─────────┤
│ District of Columbia, District of Columbia │ 11         │ 001         │ 11001   │
│ Adjuntas Municipio, Puerto Rico            │ 72         │ 001         │ 72001   │
│ Aguada Municipio, Puerto Rico              │ 72         │ 003         │ 72003   │
│ Aguadilla Municipio, Puerto Rico           │ 72         │ 005         │ 72005   │
│ Aguas Buenas Municipio, Puerto Rico        │ 72         │ 007         │ 72007   │
│ Aibonito Municipio, Puerto Rico            │ 72         │ 009         │ 72009   │
│ Añasco Municipio, Puerto Rico              │ 72         │ 011         │ 72011   │
│ Arecibo Municipio, Puerto Rico             │ 72         │ 013         │ 72

### CMS geographies without an ACS match

In [15]:
duckdb.sql('''
SELECT
    cms.county_name,
    cms.fips
FROM (
    SELECT DISTINCT
        county_name,
        fips
    FROM cms_clean
) AS cms
LEFT JOIN acs_clean AS acs
    ON cms.fips = acs.fips
WHERE acs.fips IS NULL
ORDER BY cms.fips;
''')

┌───────────────┬─────────┐
│  county_name  │  fips   │
│    varchar    │ varchar │
├───────────────┼─────────┤
│ CT-Fairfield  │ 09001   │
│ CT-Hartford   │ 09003   │
│ CT-Litchfield │ 09005   │
│ CT-Middlesex  │ 09007   │
│ CT-New Haven  │ 09009   │
│ CT-New London │ 09011   │
│ CT-Tolland    │ 09013   │
│ CT-Windham    │ 09015   │
│ VI-St. Croix  │ 78010   │
│ VI-St. John   │ 78020   │
│ VI-St. Thomas │ 78030   │
└───────────────┴─────────┘
  11 rows       2 columns

### Connecticut geography change

CMS uses the historical 8 Connecticut counties in 2020–2021 and the newer 9 planning regions in 2022–2024. Because the 2024 ACS uses the newer planning regions, Connecticut is excluded rather than mixing different geographic definitions across the five-year CMS period.

In [16]:
duckdb.sql('''
SELECT
    year,
    COUNT(DISTINCT fips) AS ct_geographies
FROM cms_clean
WHERE fips LIKE '09%'
GROUP BY year
ORDER BY year;
''')

┌───────┬────────────────┐
│ year  │ ct_geographies │
│ int64 │     int64      │
├───────┼────────────────┤
│  2020 │              8 │
│  2021 │              8 │
│  2022 │              9 │
│  2023 │              9 │
│  2024 │              9 │
└───────┴────────────────┘

## 5. Aggregate CMS to One Row per Geography

CMS contains annual records from 2020–2024, while the ACS is already a pooled five-year estimate.

For CMS:

- `avg_annual_om_beneficiaries` is the average annual Original Medicare beneficiary count.
- Age, sex percentages, and utilization measures use **Original Medicare beneficiary-weighted averages**.
- For each metric, years with a missing metric or missing beneficiary count are excluded from both the numerator and denominator.
- Connecticut (`09` FIPS prefix) is excluded before aggregation.

In [17]:
duckdb.sql('''
CREATE OR REPLACE VIEW cms_agg AS
SELECT
    fips,
    county_name,

    AVG(om_beneficiaries) AS avg_annual_om_beneficiaries,

    SUM(avg_age * om_beneficiaries)
        FILTER (WHERE avg_age IS NOT NULL AND om_beneficiaries IS NOT NULL)
    /
    SUM(om_beneficiaries)
        FILTER (WHERE avg_age IS NOT NULL AND om_beneficiaries IS NOT NULL)
        AS avg_age,

    SUM(female_pct * om_beneficiaries)
        FILTER (WHERE female_pct IS NOT NULL AND om_beneficiaries IS NOT NULL)
    /
    SUM(om_beneficiaries)
        FILTER (WHERE female_pct IS NOT NULL AND om_beneficiaries IS NOT NULL)
        AS female_pct,

    SUM(male_pct * om_beneficiaries)
        FILTER (WHERE male_pct IS NOT NULL AND om_beneficiaries IS NOT NULL)
    /
    SUM(om_beneficiaries)
        FILTER (WHERE male_pct IS NOT NULL AND om_beneficiaries IS NOT NULL)
        AS male_pct,

    SUM(inpatient_stays_per_1000 * om_beneficiaries)
        FILTER (WHERE inpatient_stays_per_1000 IS NOT NULL AND om_beneficiaries IS NOT NULL)
    /
    SUM(om_beneficiaries)
        FILTER (WHERE inpatient_stays_per_1000 IS NOT NULL AND om_beneficiaries IS NOT NULL)
        AS inpatient_stays_per_1000,

    SUM(outpatient_visits_per_1000 * om_beneficiaries)
        FILTER (WHERE outpatient_visits_per_1000 IS NOT NULL AND om_beneficiaries IS NOT NULL)
    /
    SUM(om_beneficiaries)
        FILTER (WHERE outpatient_visits_per_1000 IS NOT NULL AND om_beneficiaries IS NOT NULL)
        AS outpatient_visits_per_1000,

    SUM(er_visits_per_1000 * om_beneficiaries)
        FILTER (WHERE er_visits_per_1000 IS NOT NULL AND om_beneficiaries IS NOT NULL)
    /
    SUM(om_beneficiaries)
        FILTER (WHERE er_visits_per_1000 IS NOT NULL AND om_beneficiaries IS NOT NULL)
        AS er_visits_per_1000

FROM cms_clean
WHERE fips NOT LIKE '09%'
GROUP BY
    fips,
    county_name;
''')

### Validate aggregated CMS data

In [18]:
duckdb.sql('''
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT fips) AS unique_fips
FROM cms_agg;
''')

┌───────────┬─────────────┐
│ row_count │ unique_fips │
│   int64   │    int64    │
├───────────┼─────────────┤
│      3137 │        3137 │
└───────────┴─────────────┘

In [19]:
duckdb.sql('''
SELECT
    COUNT(*) FILTER (WHERE avg_annual_om_beneficiaries IS NULL) AS missing_beneficiaries,
    COUNT(*) FILTER (WHERE avg_age IS NULL) AS missing_avg_age,
    COUNT(*) FILTER (WHERE female_pct IS NULL) AS missing_female,
    COUNT(*) FILTER (WHERE male_pct IS NULL) AS missing_male,
    COUNT(*) FILTER (WHERE inpatient_stays_per_1000 IS NULL) AS missing_inpatient,
    COUNT(*) FILTER (WHERE outpatient_visits_per_1000 IS NULL) AS missing_outpatient,
    COUNT(*) FILTER (WHERE er_visits_per_1000 IS NULL) AS missing_er
FROM cms_agg;
''')

┌───────────────────────┬─────────────────┬────────────────┬──────────────┬───────────────────┬────────────────────┬────────────┐
│ missing_beneficiaries │ missing_avg_age │ missing_female │ missing_male │ missing_inpatient │ missing_outpatient │ missing_er │
│         int64         │      int64      │     int64      │    int64     │       int64       │       int64        │   int64    │
├───────────────────────┼─────────────────┼────────────────┼──────────────┼───────────────────┼────────────────────┼────────────┤
│                     1 │               1 │              3 │            3 │                 4 │                  2 │          3 │
└───────────────────────┴─────────────────┴────────────────┴──────────────┴───────────────────┴────────────────────┴────────────┘

In [20]:
duckdb.sql('''
SELECT
    *
FROM cms_agg
WHERE avg_annual_om_beneficiaries IS NULL
   OR avg_age IS NULL
   OR female_pct IS NULL
   OR male_pct IS NULL
   OR inpatient_stays_per_1000 IS NULL
   OR outpatient_visits_per_1000 IS NULL
   OR er_visits_per_1000 IS NULL
ORDER BY fips;
''')

┌─────────┬─────────────┬─────────────────────────────┬───────────────────┬────────────┬─────────────────────┬──────────────────────────┬────────────────────────────┬────────────────────┐
│  fips   │ county_name │ avg_annual_om_beneficiaries │      avg_age      │ female_pct │      male_pct       │ inpatient_stays_per_1000 │ outpatient_visits_per_1000 │ er_visits_per_1000 │
│ varchar │   varchar   │           double            │      double       │   double   │       double        │          double          │           double           │       double       │
├─────────┼─────────────┼─────────────────────────────┼───────────────────┼────────────┼─────────────────────┼──────────────────────────┼────────────────────────────┼────────────────────┤
│ 15005   │ HI-Kalawao  │                        NULL │              NULL │       NULL │                NULL │                     NULL │                       NULL │               NULL │
│ 48261   │ TX-Kenedy   │                        24.6 │ 72.8

## 6. Merge CMS and ACS

An `INNER JOIN` keeps only geographies represented in both sources. At this stage:

- Connecticut has already been excluded from CMS.
- ACS-only geographies such as Puerto Rico and D.C. do not enter the final dataset.
- CMS-only Virgin Islands geographies do not enter the final dataset.

In [21]:
duckdb.sql('''
CREATE OR REPLACE VIEW analysis_data AS
SELECT
    cms.fips,
    cms.county_name,
    cms.avg_annual_om_beneficiaries,
    cms.avg_age,
    cms.female_pct,
    cms.male_pct,
    cms.inpatient_stays_per_1000,
    cms.outpatient_visits_per_1000,
    cms.er_visits_per_1000,
    acs.median_household_income,
    acs.poverty_rate,
    acs.bachelors_or_higher_pct
FROM cms_agg AS cms
INNER JOIN acs_clean AS acs
    ON cms.fips = acs.fips;
''')

### Final dataset validation

In [22]:
duckdb.sql('''
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT fips) AS unique_fips
FROM analysis_data;
''')

┌───────────┬─────────────┐
│ row_count │ unique_fips │
│   int64   │    int64    │
├───────────┼─────────────┤
│      3134 │        3134 │
└───────────┴─────────────┘

In [23]:
duckdb.sql('''
SELECT
    COUNT(*) FILTER (WHERE avg_annual_om_beneficiaries IS NULL) AS missing_beneficiaries,
    COUNT(*) FILTER (WHERE avg_age IS NULL) AS missing_avg_age,
    COUNT(*) FILTER (WHERE female_pct IS NULL) AS missing_female,
    COUNT(*) FILTER (WHERE male_pct IS NULL) AS missing_male,
    COUNT(*) FILTER (WHERE inpatient_stays_per_1000 IS NULL) AS missing_inpatient,
    COUNT(*) FILTER (WHERE outpatient_visits_per_1000 IS NULL) AS missing_outpatient,
    COUNT(*) FILTER (WHERE er_visits_per_1000 IS NULL) AS missing_er,
    COUNT(*) FILTER (WHERE median_household_income IS NULL) AS missing_income,
    COUNT(*) FILTER (WHERE poverty_rate IS NULL) AS missing_poverty,
    COUNT(*) FILTER (WHERE bachelors_or_higher_pct IS NULL) AS missing_education
FROM analysis_data;
''')

┌───────────────────────┬─────────────────┬────────────────┬──────────────┬───────────────────┬────────────────────┬────────────┬────────────────┬─────────────────┬───────────────────┐
│ missing_beneficiaries │ missing_avg_age │ missing_female │ missing_male │ missing_inpatient │ missing_outpatient │ missing_er │ missing_income │ missing_poverty │ missing_education │
│         int64         │      int64      │     int64      │    int64     │       int64       │       int64        │   int64    │     int64      │      int64      │       int64       │
├───────────────────────┼─────────────────┼────────────────┼──────────────┼───────────────────┼────────────────────┼────────────┼────────────────┼─────────────────┼───────────────────┤
│                     1 │               1 │              3 │            3 │                 4 │                  2 │          3 │              1 │               0 │                 0 │
└───────────────────────┴─────────────────┴────────────────┴──────────────┴

In [24]:
duckdb.sql('''
SELECT
    MIN(avg_annual_om_beneficiaries) AS min_beneficiaries,
    MAX(avg_annual_om_beneficiaries) AS max_beneficiaries,
    MIN(avg_age) AS min_age,
    MAX(avg_age) AS max_age,
    MIN(female_pct) AS min_female_pct,
    MAX(female_pct) AS max_female_pct,
    MIN(male_pct) AS min_male_pct,
    MAX(male_pct) AS max_male_pct,
    MIN(inpatient_stays_per_1000) AS min_inpatient,
    MAX(inpatient_stays_per_1000) AS max_inpatient,
    MIN(outpatient_visits_per_1000) AS min_outpatient,
    MAX(outpatient_visits_per_1000) AS max_outpatient,
    MIN(er_visits_per_1000) AS min_er,
    MAX(er_visits_per_1000) AS max_er,
    MIN(median_household_income) AS min_income,
    MAX(median_household_income) AS max_income,
    MIN(poverty_rate) AS min_poverty,
    MAX(poverty_rate) AS max_poverty,
    MIN(bachelors_or_higher_pct) AS min_education,
    MAX(bachelors_or_higher_pct) AS max_education
FROM analysis_data;
''')

┌───────────────────┬───────────────────┬───────────────────┬───────────────────┬────────────────────┬────────────────────┬─────────────────────┬────────────────────┬───────────────────┬────────────────────┬───────────────────┬───────────────────┬────────────────────┬────────────────────┬────────────┬────────────┬─────────────┬─────────────┬───────────────┬───────────────┐
│ min_beneficiaries │ max_beneficiaries │      min_age      │      max_age      │   min_female_pct   │   max_female_pct   │    min_male_pct     │    max_male_pct    │   min_inpatient   │   max_inpatient    │  min_outpatient   │  max_outpatient   │       min_er       │       max_er       │ min_income │ max_income │ min_poverty │ max_poverty │ min_education │ max_education │
│      double       │      double       │      double       │      double       │       double       │       double       │       double        │       double       │      double       │       double       │      double       │      double       │ 

## 7. Export Analysis-Ready Dataset

The processed CSV contains one row per matched geography and is ready for downstream analysis and visualization.

In [25]:
duckdb.sql(f'''
COPY (
    SELECT *
    FROM analysis_data
)
TO '{OUTPUT_PATH}'
WITH (
    HEADER,
    DELIMITER ','
);
''')

In [26]:
duckdb.sql(f'''
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT fips) AS unique_fips
FROM read_csv('{OUTPUT_PATH}', AUTO_DETECT = TRUE);
''')

┌───────────┬─────────────┐
│ row_count │ unique_fips │
│   int64   │    int64    │
├───────────┼─────────────┤
│      3134 │        3134 │
└───────────┴─────────────┘

In [27]:
duckdb.sql(f'''
SELECT *
FROM read_csv('{OUTPUT_PATH}', AUTO_DETECT = TRUE)
LIMIT 5;
''')

┌─────────┬─────────────┬─────────────────────────────┬───────────────────┬────────────────────┬────────────────────┬──────────────────────────┬────────────────────────────┬────────────────────┬─────────────────────────┬──────────────┬─────────────────────────┐
│  fips   │ county_name │ avg_annual_om_beneficiaries │      avg_age      │     female_pct     │      male_pct      │ inpatient_stays_per_1000 │ outpatient_visits_per_1000 │ er_visits_per_1000 │ median_household_income │ poverty_rate │ bachelors_or_higher_pct │
│ varchar │   varchar   │           double            │      double       │       double       │       double       │          double          │           double           │       double       │          int64          │    double    │         double          │
├─────────┼─────────────┼─────────────────────────────┼───────────────────┼────────────────────┼────────────────────┼──────────────────────────┼────────────────────────────┼────────────────────┼────────────────────